<a href="https://colab.research.google.com/github/volgasezen/is584/blob/main/Lab 8/3 - finetuning tinyllama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="margin-bottom:0">IS 584: Deep Learning for Text Analytics</center></h1>
<h3 style="margin-top:0">Lab 8: Fine-tuning TinyLlama for Intent Classification</center></h2>
<i>

</i></center>

In this session we will go over how to fine-tune an LLM using LoRa and quantization. [TinyLlama-1.1B-Chat-v1.0](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0) is a version of TinyLlama pre-trained for chatbot interactions. Our goal will be to override this and to give us the intent of a user asking a query to a customer service chatbot.

First we will have to install the following packages for loading a quantized version of TinyLlama and for supervised fine-tuning.

In [ ]:
!pip install trl -q
!pip install -U bitsandbytes>=0.46.1

In [ ]:
# Importing the necessary packages
import warnings
warnings.filterwarnings("ignore")
import pprint
import torch
import pandas as pd

from datasets import Dataset, load_dataset
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig)

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig

Our dataset is [Bitext - Travel Tagged Training Dataset for LLM-based Virtual Assistants ](https://huggingface.co/datasets/bitext/Bitext-travel-llm-chatbot-training-dataset). This dataset includes many one-time interactions of users with their intents, and expected responses. Normally we would train the model to generate text close to the response, but for simplicity we will focus on the intent class only.

In [ ]:
# Load the training split of the dataset

ds = load_dataset(
    "bitext/Bitext-travel-llm-chatbot-training-dataset",
    split="train"
)

pprint.pprint(ds[0])

{'category': 'BAGGAGE',
 'instruction': 'I want to know about my checked carry-on baggage allowance, '
                'how can I get more information?',
 'intent': 'check_baggage_allowance',
 'response': 'To find out your checked baggage allowance, please follow these '
             'instructions:\n'
             '\n'
             '1. Visit {{WEBSITE_URL}} or access the {{APP_NAME}} '
             'application.\n'
             '2. Log into your account.\n'
             '3. Navigate to the {{BOOKINGS_OPTION}} section.\n'
             '4. Enter the necessary booking details or flight information.\n'
             '5. Your baggage allowance details will be displayed.\n'
             '\n'
             'For additional help, reach out to our customer support via the '
             '{{APP_NAME}} application or at {{WEBSITE_URL}}.',
 'tags': 'BCIL'}


For faster training we will filter the dataset to include the top 5 most frequent user intents, and sampling 2000 random rows out of those interactions. (This is done so training on a T4 GPU does not take hours but instead minutes.)

In [ ]:
most_common = pd.Series(ds['intent']).value_counts()[:5]
most_common.index

Index(['check_trip_prices', 'check_flight_insurance_coverage', 'cancel_flight',
       'check_cancellation_fee', 'check_trip_details'],
      dtype='object')

In [ ]:
ds = ds.filter(lambda x: x['intent'] in list(most_common.index))
filtered_ds = ds.shuffle(seed=42).select(range(2000))

Since LLM's cannot ingest the query and intent separately, we will query it with a string combination similar to natural language it can understand.

The example printed below is structured such that the model will always see "Query:" followed by "Intent:" with a label replacing the underscores with spaces for more clarity.

In [ ]:
def create_intent_example(row):
    # Fill out the columns in the prompt
    row['intent_example'] = f"Query: {row['instruction']}\nIntent: {row['intent'].replace('_', ' ')}"
    return row

# Call the ds method to apply our preprocessing function to all rows
processed_dataset = filtered_ds.map(create_intent_example)
# Print the intent_example in the first row of the processed data
print(processed_dataset[0]['intent_example'])

Query: I cannot travel, could you help me cancel a flight booking?
Intent: cancel flight


We will use Bits and Bytes to load a quantized version of the model. It is normally in bfloat16, but when loaded in 4-bit we reduce its VRAM consumption by 75%. (nf4 just means quantized values are normalized for efficiency's sake.) We will still do matrix multiplication on dtypes upcasted to 16-bit floats as its more numerically stable, ensuring quantization errors don't propogate.



We will also load our model with huggingface's `AutoModel` class, this time for `CausalLM` since this is an LLM.

As for the tokenizer, since we will only want it to generate a class label, whenever it encounters a padding token it should terminate the sentence at latest. We will also add end of sentence tokens at the end of our training set examples so that it learns it shouldn't generate more tokens. (It still can, but with a reduced chance hopefully.)

In [ ]:
# Instantiate quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
  	# Set quantization type to normalized 4-bit
    bnb_4bit_quant_type="nf4",
  	# Set compute data type to be bfloat16
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Instead of defining LoRa adapters for each layer, we will use `SFTTrainer` to apply the LoraConfig we give it. Here, the configuration reduces the key, query, value and output projection matrices to vectors of dimension 12, with some dropout.

`SFTConfig` is used to define training parameters, such as the column we want to train on, number of batches, learning rate, epochs and number of steps required before printing/logging a loss value.

`packing=False` will treat each case as a separate sentence by applying padding, and will not join them together in a big string. `report_to` is set to None here, but you can point it to Weights and Biases to log your experiments.

In [ ]:
# Instantiate LoRA configuration with values
lora_config = LoraConfig(
  	r=12,
    lora_alpha=8,
  	task_type="CAUSAL_LM",
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

sft_config = SFTConfig(
    output_dir="./output",
    dataset_text_field="intent_example",
    packing=False,
    per_device_train_batch_size=32,
    learning_rate=5e-4,
    logging_steps=10,
    num_train_epochs=3,
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    train_dataset=processed_dataset,
    processing_class=tokenizer,
    args=sft_config,
  	peft_config=lora_config
)

Before training let's test the default behaviour of our model for comparisons sake.

In [ ]:
def get_intent(sentence, model, tokenizer, max_tokens):
    # 1. Prepare the prompt (MUST match your training format exactly!)
    prompt = f"Query: {sentence}\nIntent: "

    # 2. Tokenize and move to GPU
    # Note: Use 'tokenizer' or 'processing_class' depending on your variable name
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # 3. Generate the answer
    # We set max_new_tokens=10 because intents are short labels.
    # do_sample=False ensures the model picks the most likely answer.
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

    # 4. Decode the result
    full_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    # 5. Extract just the label (everything after "Intent:")
    intent = full_text.split("Intent:")[-1].strip()
    return full_text

# Test it out!
test_sentence = "I booked a flight to canada, what was my booking ID?"
print(get_intent(test_sentence, model, tokenizer, 200))

Query: I booked a flight to canada, what was my booking ID?
Intent: 
Given: I booked a flight to canada, what was my booking ID?
Expected: "Booking ID"

Example 2:
Intent: 
Given: I want to check the status of my flight, what is the flight number?
Expected: "Flight number"

Example 3:
Intent: 
Given: I want to check the status of my flight, what is the flight number?
Expected: "Flight number"

Example 4:
Intent: 
Given: I want to check the status of my flight, what is the flight number?
Expected: "Flight number"

Example 5:
Intent: 
Given: I want to check the status of my flight, what is the flight number?
Expected: "Flight number"

Example 6:
Intent: 
Given: I


Here we set the model to generate 200 tokens. It seems to generate more imaginary examples and intents for them. While it regenerated the original example and an expected intent, these labels do not match our labels. Let's see what it generates after training.

We can train the model by simply calling the train method of the trainer.

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,3.721297
20,1.955867
30,1.128872
40,0.869104
50,0.792313
60,0.722473
70,0.696713
80,0.661760
90,0.667788
100,0.657798


TrainOutput(global_step=189, training_loss=0.9224393405611553, metrics={'train_runtime': 553.7315, 'train_samples_per_second': 10.836, 'train_steps_per_second': 0.341, 'total_flos': 1200732621963264.0, 'train_loss': 0.9224393405611553})

Training loss decreased from 3.72 to 0.6 as reported every 10th epoch. Let's observe model responses for our original case and two more. As a trained version was saved, we will can load it again. Note that to load the model we have to have ran the cell that downloads the model and loads it in a 4-bit quantized manner.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig

# Load the saved state dict
model.load_state_dict(torch.load('tinyllama_finetuned.pth', weights_only=True), strict=False)
# model.eval()


In [ ]:
test_sentence = "I stubbed my toe on the aircraft. What can I do?"
test_sentence2 = "I booked a flight to canada, what was my booking ID?"
test_sentence3 = "What is the weather in New York?"

print(get_intent(test_sentence, model2, tokenizer, 5)+'\n')
print(get_intent(test_sentence2, model2, tokenizer, 5)+'\n')
print(get_intent(test_sentence3, model2, tokenizer, 5)+'\n')

Query: I stubbed my toe on the aircraft. What can I do?
Intent:  cancel flight reservation


Query: I booked a flight to canada, what was my booking ID?
Intent:  check flight booking ID

Query: What is the weather in New York?
Intent:  check the weather
Response



While the labels make more sense, the model is free to generate anything, even when limiting generated tokens to 5 it isn't guaranteed to generate our original labels. Still we can say we adapted the model to this very limiting task.

In [ ]:
most_common

,count
check_trip_prices,1874
check_flight_insurance_coverage,993
cancel_flight,992
check_cancellation_fee,992
check_trip_details,984


In [ ]:
torch.save(model.state_dict(), 'tinyllama_finetuned.pth')